In [ ]:
# この部分はGPT4oの出力では無い。
import os

SOURCE_IMG_DIR = "cropped_64x64"
SOURCE_IMG_DIR = "cropped_64x64_permil10"
IMG_FILENAME = "223_48_1.png" # 白なし
# IMG_FILENAME = "223_48_3.png" # 白なし

IMAGE_DIR = "image_executed"

img_filename = os.path.join(SOURCE_IMG_DIR, IMG_FILENAME) 


```
pip install scikit-image
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from skimage import io, img_as_float
from torchvision import transforms
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

os.makedirs(IMAGE_DIR, exist_ok=True) # 図の保存用directory

# ハイパーパラメータの設定
learning_rate = 0.001
num_epochs = 200
display_per_epoch = 10

batch_size = 1
img_size = (64, 64)
image_mask = False
autoencoder_type = "2layers_8_32_kernelsize3"

setting = 3
if setting == 1:
    criterion_mask = False
    mask_value0 = False
    mask_value1 = False
elif setting == 2:
    criterion_mask = False
    mask_value0 = True
    mask_value1 = True
elif setting == 3:
    criterion_mask = True
    mask_value0 = True
    mask_value1 = True
else:
    ValueError("unknown setting")

imshow_vrange = (0, 1) # 画像の表示ピクセル値の範囲．imshowで用いる．

# データセットとデータローダーの定義
from skimage import io
import numpy as np
from torch.utils.data import Dataset
import torch

class ImageDataset(Dataset):
    def __init__(self, img_filename, transform=None):
        # 画像を読み込み、0-255の範囲に規格化
        self.image = io.imread(img_filename, as_gray=True)
        self.transform = transform
        
        # ピクセル値が0または255の部分をマスク
        if mask_value0:
            self.mask0 = self.image==0
        else:
            self.mask0 = np.zeros_like(self.image)
        if mask_value1:
            self.mask1 = self.image==255
        else:
            self.mask1 = np.zeros_like(self.image)
            
        self.mask = self.mask0 | self.mask1
        
        # マスク0以外の部分の最小値、マスク1以外の部分の最大値を計算
        if mask_value0:
            self.min_value = self.image[~self.mask0].min()
        else:
            self.min_value = self.image.min()
        if mask_value1:
            self.max_value = self.image[~self.mask1].max()
        else:
            self.max_value = self.image.max()

        print("min,max", self.min_value, self.max_value)
        # 画像を0-1の範囲に正規化
        self.image = (self.image - self.min_value) / (self.max_value - self.min_value)
        self.image[self.mask0] = 0.0
        self.image[self.mask1] = 1.0
        if image_mask:
            self.image[self.mask] = np.median(self.image[~self.mask]) # 中間の値にする．
            
        self.image = np.clip(self.image, 0, 1)  # 再度[0,1]に収まっているはずだが，再度行う．

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        image = self.image
        mask = self.mask
        if self.transform:
            image = self.transform(image)
            mask = torch.from_numpy(mask).unsqueeze(0).float()
        return image, mask

transform = transforms.Compose([
    transforms.ToTensor(),
])

dataset = ImageDataset(img_filename, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

if autoencoder_type == "2layers_kernelsize3":
    # モデルの定義
    class Autoencoder(nn.Module):
        def __init__(self):
            super(Autoencoder, self).__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
                nn.ReLU()
            )
            self.decoder = nn.Sequential(
                nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
                nn.ReLU(),
                nn.ConvTranspose2d(32, 1, kernel_size=3, stride=1, padding=1),
                nn.Sigmoid()
            )

        def forward(self, x):
            x = self.encoder(x)
            x = self.decoder(x)
            return x

elif autoencoder_type == "2layers_8_32_kernelsize3":
    class Autoencoder(nn.Module):
        def __init__(self):
            super(Autoencoder, self).__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(1, 8, kernel_size=3, padding=1), # 第一層: 1x8x???
                nn.ReLU(),
                nn.Conv2d(8, 32, kernel_size=3, padding=1), # 第二層: 8x32x???
                nn.ReLU()
            )
            self.decoder = nn.Sequential(
                nn.Conv2d(32, 8, kernel_size=3, padding=1), # デコーダの対応する層も修正
                nn.ReLU(),
                nn.Conv2d(8, 1, kernel_size=3, padding=1), # 出力を元の1チャネルに戻す
                nn.Sigmoid()
            )

        def forward(self, x):
            x = self.encoder(x)
            x = self.decoder(x)
            return x

elif autoencoder_type == "3layers_kernelsize3":
    class Autoencoder(nn.Module):
        def __init__(self):
            super(Autoencoder, self).__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
                nn.ReLU(),
                nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
                nn.ReLU()
            )
            self.decoder = nn.Sequential(
                nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
                nn.ReLU(),
                nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
                nn.ReLU(),
                nn.ConvTranspose2d(32, 1, kernel_size=3, stride=1, padding=1),
                nn.Sigmoid()
            )

        def forward(self, x):
            x = self.encoder(x)
            x = self.decoder(x)
            return x

elif autoencoder_type == "3layers_kernelsize5":
    # モデルの定義
    class Autoencoder(nn.Module):
        def __init__(self):
            super(Autoencoder, self).__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),  # 大きなカーネルサイズ
                nn.ReLU(),
                nn.MaxPool2d(2, stride=2, padding=0),  # プーリング層でダウンサンプリング
                nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
                nn.ReLU(),
                nn.MaxPool2d(2, stride=2, padding=0),  # さらにダウンサンプリング
                nn.Conv2d(64, 128, kernel_size=5, stride=1, padding=2),
                nn.ReLU(),
                nn.MaxPool2d(2, stride=2, padding=0)  # さらにダウンサンプリング
            )
            self.decoder = nn.Sequential(
                nn.ConvTranspose2d(128, 64, kernel_size=5, stride=2, padding=2, output_padding=1),
                nn.ReLU(),
                nn.ConvTranspose2d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1),
                nn.ReLU(),
                nn.ConvTranspose2d(32, 1, kernel_size=5, stride=2, padding=2, output_padding=1),
                nn.Sigmoid()
            )

        def forward(self, x):
            x = self.encoder(x)
            x = self.decoder(x)
            return x
else:
    raise ValueError('unknown autoencoder_type')

model = Autoencoder()

# 損失関数とオプティマイザーの定義
if criterion_mask:
    criterion = nn.MSELoss(reduction='none')  # 損失をピクセルレベルで計算するため'reduction'を'none'に設定
else:
    criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# トレーニングループ
losses = []
psnrs = []
ssims = []

def plot_histograms(img, denoised_img, epoch, img_name):
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.hist(img.flatten(), bins=50, color='blue', alpha=0.7, label='Original', density=True)
    ax.hist(denoised_img.flatten(), bins=50, color='red', alpha=0.7, label='Denoised', density=True)
    ax.set_title(f'Epoch {epoch+1}: Pixel Value Distribution')
    ax.legend()
    plt.savefig(os.path.join(IMAGE_DIR, f"DOS_from_{img_name}_epoch_{epoch+1}.png"))
    plt.show()

for epoch in range(num_epochs):
    for data in dataloader:
        img, mask = data
        img = img.float()
        mask = mask.float()
        # フォワードパス
        output = model(img)

        # 損失計算 
        if criterion_mask:
            mask_inverted = 1 - mask  # マスクを反転
            loss = criterion(output, img)
            loss = loss * mask_inverted  # マスク部分の損失を0にする
            loss = (loss.sum(dim=(1, 2, 3)) / mask_inverted.sum(dim=(1, 2, 3))).mean()  # マスク外のピクセルのみで平均化
        else:
            loss = criterion(output, img)

        # PSNRとSSIMの計算
        output_img = output.detach().numpy().squeeze()
        input_img = img.detach().numpy().squeeze()
        mask_img = mask.detach().numpy().squeeze()
        output_img = np.clip(output_img, 0, 1)
        input_img = np.clip(input_img, 0, 1)
        psnr = peak_signal_noise_ratio(input_img, output_img, data_range=1)
        ssim = structural_similarity(input_img, output_img, data_range=1)

        # バックプロパゲーション
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        psnrs.append(psnr)
        ssims.append(ssim)

    if (epoch + 1) % display_per_epoch == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, PSNR: {psnr:.4f}, SSIM: {ssim:.4f}')
        # 画像の比較表示
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(input_img, cmap='gray', vmin=imshow_vrange[0], vmax=imshow_vrange[1])
        axes[0].set_title('Original Image')
        axes[1].imshow(output_img, cmap='gray', vmin=imshow_vrange[0], vmax=imshow_vrange[1])
        axes[1].set_title('Denoised Image')
        # axes[2].imshow(mask_img, cmap='gray', vmin=0, vmax=1)
        # axes[2].set_title('Mask Image')        
        img_name = IMG_FILENAME[:] # 実体をコピー
        img_name = img_name.replace(".","_")
        fig.tight_layout()
        fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05)
        plt.savefig(os.path.join(IMAGE_DIR, f"riginal_vs_denoised_image_from_{img_name}_epoch_{epoch+1}.png"))
        plt.show()

        # ピクセル値のヒストグラムを表示
        plot_histograms(input_img, output_img, epoch, img_name)

# 損失関数の図示
plt.figure()
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Epoch vs Loss')
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, f"epoch_vs_loss_from_{img_name}.png"))
plt.show()

# PSNR and SSIMの図示
plt.figure()
plt.plot(psnrs, label='PSNR')
plt.xlabel('Epoch')
plt.ylabel('PSNR')
plt.legend()
plt.title('Epoch vs PSNR')
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, f"epoch_vs_PSNR_from_{img_name}.png"))
plt.show()

plt.figure()
plt.plot(ssims, label='SSIM')
plt.xlabel('Epoch')
plt.ylabel('SSIM')
plt.legend()
plt.title('Epoch vs SSIM')
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, f"epoch_vs_SSIM_from_{img_name}.png"))
plt.show()

テストデータが無いので，epochに対して値が一方向に変化する．